In [ ]:
import requests
import json

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "geojson",
    "starttime": "2026-07-01",
    "endtime": "2026-08-01",
    "minmagnitude": "2.5"
}

try:
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    print(len(data["features"]))
except requests.exceptions.RequestException as e:
    print(f"API Request failed: {e}")


2643


In [3]:
from pathlib import Path

event_ids = [f["id"] for f in data["features"]]

raw_dir = Path.cwd().parent / "data" / "raw"
output_file = raw_dir / "extracted_ids.txt"
output_file.write_text("\n".join(event_ids), encoding="utf-8")

print(f"Saved {len(event_ids)} IDs to {output_file.name}")

Saved 2643 IDs to extracted_ids.txt


In [4]:
api_file = Path.cwd().parent / "data" / "raw" / "earthquakes_raw.json"
with open(api_file, mode="w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

print(f"Saved raw API data to {api_file.name}")

Saved raw API data to earthquakes_raw.json


In [5]:
import csv

csv_path = Path.cwd().parent / "data" / "raw" / "regional_sensor_log.csv"

sensor_rows = []
with open(csv_path, mode="r", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    print(f"Detected Headers: {reader.fieldnames}")
    for row in reader:
        sensor_rows.append(row)

print(f"Total rows loaded: {len(sensor_rows)}")
print("\nFirst row sample:")
print(sensor_rows[0])

Detected Headers: ['event_id', 'station_network', 'local_claims_filed']
Total rows loaded: 2611

First row sample:
{'event_id': 'us7000t1ii', 'station_network': 'SCEDC-01', 'local_claims_filed': '12'}


In [6]:
import json
from pathlib import Path

raw_dir = Path.cwd().parent / "data" / "raw"
json_path = raw_dir / "earthquakes_raw.json"

with open(json_path, mode="r", encoding="utf-8") as f:
    api_data = json.load(f)

records = api_data["features"]
print(f"Total records loaded: {len(records)}\n")

def print_leaf_types(node, current_path="root"):
    if isinstance(node, dict):
        for key, value in node.items():
            print_leaf_types(value, f"{current_path}['{key}']")
    elif isinstance(node, list):
        if len(node) > 0:
            
            print_leaf_types(node[0], f"{current_path}[0]")
    else:
        print(f"{current_path} -> {type(node).__name__}")

print("--- Structural Audit of First Record ---")
if records:
    print_leaf_types(records[0])

Total records loaded: 2643

--- Structural Audit of First Record ---
root['type'] -> str
root['properties']['mag'] -> float
root['properties']['place'] -> str
root['properties']['time'] -> int
root['properties']['updated'] -> int
root['properties']['tz'] -> NoneType
root['properties']['url'] -> str
root['properties']['detail'] -> str
root['properties']['felt'] -> NoneType
root['properties']['cdi'] -> NoneType
root['properties']['mmi'] -> NoneType
root['properties']['alert'] -> NoneType
root['properties']['status'] -> str
root['properties']['tsunami'] -> int
root['properties']['sig'] -> int
root['properties']['net'] -> str
root['properties']['code'] -> str
root['properties']['ids'] -> str
root['properties']['sources'] -> str
root['properties']['types'] -> str
root['properties']['nst'] -> int
root['properties']['dmin'] -> float
root['properties']['rms'] -> float
root['properties']['gap'] -> int
root['properties']['magType'] -> str
root['properties']['type'] -> str
root['properties']['tit

In [7]:
mag_count = 0
mag_min = None
mag_max = None
mag_sum = 0.0

depth_count = 0
depth_min = None
depth_max = None
depth_sum = 0.0
null_fields = ["felt", "cdi", "mmi", "alert", "nst", "dmin", "gap"]
null_counts = {}
event_types = {}
for field in null_fields:
    null_counts[field] = 0

for record in records:
    props = record["properties"]
    geom = record["geometry"]
    coords = geom["coordinates"]

    e_type = props["type"]
    if e_type in event_types:
        event_types[e_type] = event_types[e_type] + 1
    else:
        event_types[e_type] = 1
    
    for field in null_fields:
        if props.get(field) is None:
            null_counts[field] = null_counts[field] + 1

    m = props["mag"]
    if m is not None:
        if mag_min is None or m < mag_min:
            mag_min = m
        if mag_max is None or m > mag_max:
            mag_max = m
        mag_sum = mag_sum + m
        mag_count = mag_count + 1

    if len(coords) > 2:
        d = coords[2]
        if d is not None:
            if depth_min is None or d < depth_min:
                depth_min = d
            if depth_max is None or d > depth_max:
                depth_max = d
            depth_sum = depth_sum + d
            depth_count = depth_count + 1

mag_mean = (mag_sum / mag_count) if mag_count > 0 else 0.0
depth_mean = (depth_sum / depth_count) if depth_count > 0 else 0.0
print(f"Magnitude - Min: {mag_min}, Max: {mag_max}, Mean: {mag_mean:.2f}")
print(f"Depth (km) - Min: {depth_min}, Max: {depth_max}, Mean: {depth_mean:.2f}")

print("\n--- Event Type Breakdown ---")
for t, count in event_types.items():
    print(f"  Type '{t}': {count} records")

print("\n--- Missing Value (Null) Rates ---")
total_recs = len(records)
for field, count in null_counts.items():
    percentage = (count / total_recs) * 100
    print(f"  {field}: {count} nulls ({percentage:.1f}%)")


Magnitude - Min: 2.5, Max: 7.3, Mean: 3.87
Depth (km) - Min: -2.85, Max: 670.304, Mean: 55.40

--- Event Type Breakdown ---
  Type 'earthquake': 2639 records
  Type 'explosion': 1 records
  Type 'mining explosion': 1 records
  Type 'experimental explosion': 2 records

--- Missing Value (Null) Rates ---
  felt: 2303 nulls (87.1%)
  cdi: 2303 nulls (87.1%)
  mmi: 2413 nulls (91.3%)
  alert: 2557 nulls (96.7%)
  nst: 0 nulls (0.0%)
  dmin: 1 nulls (0.0%)
  gap: 0 nulls (0.0%)


In [ ]:
import csv
from pathlib import Path

sensor_dict = {}
for row in sensor_rows:
    sensor_dict[row["event_id"]] = row

gaps, dmins, nsts = [], [], []
for record in records:
    props = record["properties"]
    if props.get("type") == "earthquake":
        if props.get("gap") is not None:
            gaps.append(props.get("gap"))
        if props.get("dmin") is not None:
            dmins.append(props.get("dmin"))
        if props.get("nst") is not None:
            nsts.append(props.get("nst"))

gaps.sort()
dmins.sort()
nsts.sort()

median_gap = gaps[len(gaps) // 2] if gaps else 0.0
median_dmin = dmins[len(dmins) // 2] if dmins else 0.0
median_nst = nsts[len(nsts) // 2] if nsts else 0.0

clean_records = []
mag_values = []

In [ ]:
for record in records:
    props = record["properties"]
    geom = record["geometry"]
    coords = geom["coordinates"]
    
    if props.get("type") != "earthquake":
        continue
        
    mag = props.get("mag")
    if mag is None:
        continue
        
    place = props.get("place", "")
    if place is None:
        place = ""
        
    if " of " in place:
        parts = place.split(" of ")
        region = parts[1]
    else:
        region = place
        
    depth_km = coords[2] if (len(coords) > 2 and coords[2] is not None) else 0.0
    if depth_km < 70:
        depth_category = "shallow"
    elif depth_km <= 300:
        depth_category = "intermediate"
    else:
        depth_category = "deep"
        
    felt = props.get("felt")
    if felt is None:
        felt = 0
        
    gap = props.get("gap")
    if gap is None:
        gap = median_gap

    dmin = props.get("dmin")
    if dmin is None:
        dmin = median_dmin

    nst = props.get("nst")
    if nst is None:
        nst = median_nst
        
    significant = 1 if mag >= 5.0 else 0
    sig = props.get("sig", 0)
        
    event_id = record["id"]
    if event_id in sensor_dict:
        log_data = sensor_dict[event_id]
        station_net = log_data["station_network"]
        claims = log_data["local_claims_filed"]
    else:
        station_net = "unknown"
        claims = "0"
        
    clean_row = {
        "event_id": event_id,
        "mag": mag,
        "depth_km": depth_km,
        "depth_category": depth_category,
        "felt": felt,
        "gap": gap,
        "dmin": dmin,
        "nst": nst,
        "significant": significant,
        "sig": sig if sig is not None else 0,
        "region": region,
        "station_network": station_net,
        "local_claims_filed": claims
    }
    
    clean_records.append(clean_row)
    mag_values.append(mag)

In [ ]:
sig_1_list = [r["sig"] for r in clean_records if r["significant"] == 1]
sig_0_list = [r["sig"] for r in clean_records if r["significant"] == 0]
avg_sig_1 = sum(sig_1_list) / len(sig_1_list) if len(sig_1_list) > 0 else 0
avg_sig_0 = sum(sig_0_list) / len(sig_0_list) if len(sig_0_list) > 0 else 0
print(f"Validation: Avg Sig (Significant=1): {avg_sig_1:.2f} | Avg Sig (Significant=0): {avg_sig_0:.2f}")

n_total = len(clean_records)
n_flagged = len(sig_1_list)
workload_reduction = (1 - (n_flagged / n_total)) * 100 if n_total > 0 else 0.0
print(f"ROI Metrics: n_total={n_total}, n_flagged={n_flagged}, Workload Reduction={workload_reduction:.2f}%")

v_min = min(mag_values) if mag_values else 0.0
v_max = max(mag_values) if mag_values else 1.0

for row in clean_records:
    original_mag = row["mag"]
    if v_max - v_min > 0:
        row["scaled_mag"] = (original_mag - v_min) / (v_max - v_min)
    else:
        row["scaled_mag"] = 0.0

output_file = Path.cwd().parent / "data" / "processed" / "clean_data.csv"
fieldnames = [
    "event_id", "mag", "scaled_mag", "depth_km", "depth_category", 
    "felt", "gap", "dmin", "nst", "significant", "region", 
    "station_network", "local_claims_filed"
]

with open(output_file, mode="w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()
    for row in clean_records:
        writer.writerow(row)

print(f"Processed {len(clean_records)} records saved to {output_file.name}")

Validation: Avg Sig (Significant=1): 436.42 | Avg Sig (Significant=0): 225.92
Processed 2639 records saved to clean_data.csv
